In [7]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Buspulse-OnTime-Analysis")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

stop_visits = spark.read.parquet("stop_visits.parquet")
routes = spark.read.parquet("gtfs_routes.parquet")

# Inspect the data
stop_visits.printSchema()
stop_visits.select(
    "vehicle_mode",
    "call_type",
    "aimed_arrival",
    "actual_arrival"
).show(10, truncate=False)

print("Total rows:", stop_visits.count())


root
 |-- snapshot_time: timestamp (nullable = true)
 |-- recorded_at: string (nullable = true)
 |-- journey_ref: string (nullable = true)
 |-- data_frame_ref: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- direction_ref: string (nullable = true)
 |-- vehicle_mode: string (nullable = true)
 |-- operator_ref: string (nullable = true)
 |-- monitored: boolean (nullable = true)
 |-- journey_cancelled: boolean (nullable = true)
 |-- call_type: string (nullable = true)
 |-- stop_order: integer (nullable = true)
 |-- stop_ref: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- aimed_arrival: string (nullable = true)
 |-- expected_arrival: string (nullable = true)
 |-- actual_arrival: string (nullable = true)
 |-- aimed_departure: string (nullable = true)
 |-- expected_departure: string (nullable = true)
 |-- actual_departure: string (nullable = true)
 |-- arrival_status: string (nullable = true)
 |-- departu

In [11]:

# Keep the latest snapshot for each stop visit
window_latest = (
    Window
    .partitionBy("data_frame_ref", "journey_ref", "stop_order")
    .orderBy(F.col("snapshot_time").desc())
)

visits = (
    stop_visits
    .withColumn("row_num", F.row_number().over(window_latest))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# Only buses, not ferries
visits = visits.filter(F.col("vehicle_mode") == "bus")

# Exclude cancelled services
visits = visits.filter(
    ~(
        F.coalesce(F.col("call_cancelled").cast("boolean"), F.lit(False))
        |
        F.coalesce(F.col("journey_cancelled").cast("boolean"), F.lit(False))
    )
)

# Only recorded stop visits with actual arrival timestamps
arrivals = (
    visits
    .filter(F.col("call_type") == "recorded")
    .filter(
        F.col("actual_arrival").isNotNull()
        & F.col("aimed_arrival").isNotNull()
    )
    .withColumn("aimed_ts", F.to_timestamp("aimed_arrival"))
    .withColumn("actual_ts", F.to_timestamp("actual_arrival"))
    .filter(F.col("aimed_ts").isNotNull() & F.col("actual_ts").isNotNull())
    .withColumn(
        "arrival_delay_s",
        (F.expr("unix_micros(actual_ts)") - F.expr("unix_micros(aimed_ts)")) / 1_000_000
    )
)

# Remove extreme outliers from the main analysis
# Keep the excluded rows separately for data-quality inspection
outliers = arrivals.filter(F.abs(F.col("arrival_delay_s")) > 10800)

arrivals = arrivals.filter(
    F.abs(F.col("arrival_delay_s")) <= 10800
)

print("Valid arrivals:", arrivals.count())
print("Excluded extreme outliers:", outliers.count())

arrivals.select(
    "journey_ref",
    "line_ref",
    "stop_name",
    "aimed_arrival",
    "actual_arrival",
    "arrival_delay_s"
).show(10, truncate=False)


Valid arrivals: 147874
Excluded extreme outliers: 0
+--------------------------------------------+---------------+------------------+-------------------------+---------------------------------+---------------+
|journey_ref                                 |line_ref       |stop_name         |aimed_arrival            |actual_arrival                   |arrival_delay_s|
+--------------------------------------------+---------------+------------------+-------------------------+---------------------------------+---------------+
|KOL:ServiceJourney:1006_261005064436258_1065|KOL:Line:8_1006|Varatun           |2026-10-29T23:53:00+01:00|2026-10-29T23:55:33.6715825+01:00|153.671582     |
|KOL:ServiceJourney:1006_261005064436258_1065|KOL:Line:8_1006|Rogaland Rideklubb|2026-10-30T00:14:00+01:00|2026-10-30T00:15:21.6865855+01:00|81.686585      |
|KOL:ServiceJourney:1006_261005064436258_1065|KOL:Line:8_1006|Brageveien        |2026-10-30T00:38:00+01:00|2026-10-30T00:41:33.6978115+01:00|213.697811     |


In [12]:

arrivals = arrivals.withColumn(
    "punctuality",
    F.when(F.col("arrival_delay_s") < -60, "Early")
     .when(F.col("arrival_delay_s") <= 180, "On time")
     .when(F.col("arrival_delay_s") <= 300, "Slightly late")
     .when(F.col("arrival_delay_s") <= 600, "Moderately late")
     .otherwise("Severely late")
)

arrivals = arrivals.withColumn(
    "on_time",
    F.col("arrival_delay_s").between(-60, 180)
)

arrivals.groupBy("punctuality").count().show()


+---------------+-----+
|    punctuality|count|
+---------------+-----+
|        On time|80136|
|  Slightly late|25717|
|Moderately late|25184|
|          Early| 8480|
|  Severely late| 8357|
+---------------+-----+



In [13]:

route_dim = routes.select(
    F.col("route_id").alias("line_ref"),
    F.col("route_short_name").alias("route_number"),
    F.col("route_long_name").alias("route_name")
)

arrivals = arrivals.join(
    route_dim,
    on="line_ref",
    how="left"
)

arrivals.select(
    "route_number",
    "route_name",
    "stop_name",
    "arrival_delay_s",
    "punctuality"
).show(10, truncate=False)


+------------+----------+------------------+---------------+-------------+
|route_number|route_name|stop_name         |arrival_delay_s|punctuality  |
+------------+----------+------------------+---------------+-------------+
|5           |          |Varatun           |153.671582     |On time      |
|5           |          |Rogaland Rideklubb|81.686585      |On time      |
|5           |          |Brageveien        |213.697811     |Slightly late|
|5           |          |Fjellsenden       |166.715724     |On time      |
|5           |          |Froastølveien     |180.675048     |Slightly late|
|5           |          |Føynabakken       |143.505334     |On time      |
|5           |          |Vardenes skole    |172.445464     |On time      |
|5           |          |Vølstadveien      |78.430557      |On time      |
|5           |          |UiS nord          |45.430096      |On time      |
|5           |          |UiS øst           |36.397685      |On time      |
+------------+----------+

In [14]:

route_performance = (
    arrivals
    .groupBy("route_number", "route_name")
    .agg(
        F.count("*").alias("total_arrivals"),

        F.round(
            F.avg("arrival_delay_s"), 2
        ).alias("avg_delay_seconds"),

        F.round(
            F.percentile_approx("arrival_delay_s", 0.5), 2
        ).alias("median_delay_seconds"),

        F.round(
            F.avg(F.col("on_time").cast("double")) * 100, 2
        ).alias("on_time_percentage"),

        F.round(
            F.avg(
                (F.col("arrival_delay_s") > 600).cast("double")
            ) * 100, 2
        ).alias("severely_late_percentage")
    )
    .orderBy(F.desc("avg_delay_seconds"))
)

route_performance.show(50, truncate=False)


+------------+----------+--------------+-----------------+--------------------+------------------+------------------------+
|route_number|route_name|total_arrivals|avg_delay_seconds|median_delay_seconds|on_time_percentage|severely_late_percentage|
+------------+----------+--------------+-----------------+--------------------+------------------+------------------------+
|SK1411      |          |4             |3340.35          |3010.35             |0.0               |100.0                   |
|SK1202      |          |2             |1476.07          |312.3               |0.0               |50.0                    |
|SK0113      |          |28            |1308.4           |158.73              |28.57             |35.71                   |
|SK3404      |          |24            |1173.8           |-0.31               |45.83             |29.17                   |
|SK1410      |          |4             |1153.85          |898.85              |0.0               |100.0                   |
|246    

In [15]:

arrivals = (
    arrivals
    .withColumn(
        "local_aimed_arrival",
        F.from_utc_timestamp(
            F.col("aimed_arrival"),
            "Europe/Oslo"
        )
    )
    .withColumn("hour", F.hour("local_aimed_arrival"))
    .withColumn("service_date", F.to_date("local_aimed_arrival"))
    .withColumn("weekday", F.date_format("local_aimed_arrival", "EEEE"))
)

hourly_performance = (
    arrivals
    .groupBy("hour")
    .agg(
        F.count("*").alias("total_arrivals"),
        F.round(F.avg("arrival_delay_s"), 2).alias("avg_delay_seconds"),
        F.round(
            F.avg(F.col("on_time").cast("double")) * 100, 2
        ).alias("on_time_percentage")
    )
    .orderBy("hour")
)

hourly_performance.show(24)


+----+--------------+-----------------+------------------+
|hour|total_arrivals|avg_delay_seconds|on_time_percentage|
+----+--------------+-----------------+------------------+
|   0|          3229|           126.96|             68.16|
|   1|           593|            76.82|             67.79|
|   2|           398|            86.69|              69.6|
|   3|           329|            93.79|             72.95|
|   4|           353|            57.96|             81.02|
|   5|          1234|            98.36|             78.04|
|   6|          7770|           126.45|              65.1|
|   7|         12556|           196.41|             50.88|
|   8|         10226|           188.39|             52.38|
|   9|          8066|           108.17|             64.26|
|  10|          7769|           127.17|             65.59|
|  11|          7722|           141.79|             64.39|
|  12|          8108|           153.22|             57.67|
|  13|          9011|           203.27|             48.8

In [16]:

weekday_performance = (
    arrivals
    .groupBy("weekday")
    .agg(
        F.count("*").alias("total_arrivals"),
        F.round(F.avg("arrival_delay_s"), 2).alias("avg_delay_seconds"),
        F.round(
            F.avg(F.col("on_time").cast("double")) * 100, 2
        ).alias("on_time_percentage")
    )
)

weekday_performance.show()


+--------+--------------+-----------------+------------------+
| weekday|total_arrivals|avg_delay_seconds|on_time_percentage|
+--------+--------------+-----------------+------------------+
|  Friday|        145462|           197.99|             53.98|
|Thursday|            41|          5826.25|             29.27|
|Saturday|          2371|            80.66|             67.74|
+--------+--------------+-----------------+------------------+



In [17]:

stop_performance = (
    arrivals
    .groupBy("stop_ref", "stop_name", "route_number")
    .agg(
        F.count("*").alias("total_arrivals"),
        F.round(F.avg("arrival_delay_s"), 2).alias("avg_delay_seconds"),
        F.round(
            F.avg(F.col("on_time").cast("double")) * 100, 2
        ).alias("on_time_percentage")
    )
    .filter(F.col("total_arrivals") >= 10)
    .orderBy(F.desc("avg_delay_seconds"))
)

stop_performance.show(30, truncate=False)


+---------------+-------------------+------------+--------------+-----------------+------------------+
|stop_ref       |stop_name          |route_number|total_arrivals|avg_delay_seconds|on_time_percentage|
+---------------+-------------------+------------+--------------+-----------------+------------------+
|NSR:Quay:102619|Aksdal bussterminal|246         |10            |1165.39          |20.0              |
|NSR:Quay:48514 |Austbø vest        |X74         |17            |623.48           |11.76             |
|NSR:Quay:45808 |Skolebryggå        |X74         |17            |617.67           |11.76             |
|NSR:Quay:46026 |Hunstein           |X74         |17            |605.67           |11.76             |
|NSR:Quay:47556 |Sævelandsvik       |209         |21            |605.53           |38.1              |
|NSR:Quay:44970 |UiS nord           |70          |20            |604.08           |0.0               |
|NSR:Quay:47986 |Tjøsvoll           |209         |21            |592.32  